# 04 Results Figures

## Why This Notebook Exists
This notebook assembles a clean, shareable figure/table pack from the latest strategy runs. Use it before publishing updates or sharing interim results.

## Output Locations
- Figures: `reports/figures/final/`
- Tables: `reports/tables/final/`


In [ ]:
# Standard notebook bootstrap.
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.experiments.run_baseline import run as run_baseline
from src.experiments.run_regimes import run as run_regimes
from src.experiments.run_stress import run as run_stress
from src.utils.io import load_config


## Run Controls
- Set `CONFIG_PATH` to your desired scenario file.
- This notebook intentionally reruns baseline/regimes/stress so outputs are synchronized.


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG_PATH)

# Regenerate core artifacts so final figures and tables are internally consistent.
backtests = run_baseline(str(CONFIG_PATH))
labeled_by_pair = run_regimes(str(CONFIG_PATH))
stress = run_stress(str(CONFIG_PATH))

final_fig_dir = ROOT / "reports" / "figures" / "final"
final_tbl_dir = ROOT / "reports" / "tables" / "final"
final_fig_dir.mkdir(parents=True, exist_ok=True)
final_tbl_dir.mkdir(parents=True, exist_ok=True)
print("Saving final figures to:", final_fig_dir)
print("Saving final tables to:", final_tbl_dir)


## Figure Pack
1. Baseline equity curves
2. Regime heatmaps (mean net return)
   - Axis key: `x = vol_bin` (volatility quantile), `y = mu_bin` (drift quantile)
   - Quartile key: `Q1` lowest 25%, `Q2` 25-50%, `Q3` 50-75%, `Q4` highest 25%
3. Historical stress window returns
4. Monte Carlo total-return distributions


In [ ]:
# Figure 1: baseline equity comparison.
fig, ax = plt.subplots(figsize=(10, 5))
for name, bt in backtests.items():
    ax.plot(bt.index, bt["equity"], label=name, lw=1.7)
ax.set_title("Figure 1. Baseline equity curves")
ax.set_ylabel("Equity")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(final_fig_dir / "fig1_equity_curves.png", dpi=200)
plt.show()


In [ ]:
# Figure 2: regime heatmaps for each pair.
pair_names = list(labeled_by_pair.keys())
fig, axes = plt.subplots(1, len(pair_names), figsize=(6 * len(pair_names), 5), squeeze=False)
for i, name in enumerate(pair_names):
    df = labeled_by_pair[name]
    heat = df.pivot_table(index="mu_bin", columns="vol_bin", values="net_ret", aggfunc="mean", observed=False)
    vals = heat.values.astype(float)
    im = axes[0, i].imshow(vals, cmap="RdYlBu_r", aspect="auto")
    axes[0, i].set_title(f"{name}: mean net return")
    axes[0, i].set_xticks(range(heat.shape[1]))
    axes[0, i].set_yticks(range(heat.shape[0]))
    axes[0, i].set_xticklabels(heat.columns)
    axes[0, i].set_yticklabels(heat.index)
    axes[0, i].set_xlabel("Volatility Quantile Bin (vol_bin)")
    axes[0, i].set_ylabel("Drift Quantile Bin (mu_bin)")

    # Annotate each cell for readability in static exports.
    for r in range(heat.shape[0]):
        for c in range(heat.shape[1]):
            if np.isfinite(vals[r, c]):
                axes[0, i].text(c, r, f"{vals[r, c]:.4f}", ha="center", va="center", fontsize=8)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
cbar.set_label("Mean Daily Net Return")
fig.text(
    0.5,
    0.01,
    "Key: Q1=lowest quartile, Q2=25-50%, Q3=50-75%, Q4=highest quartile",
    ha="center",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(final_fig_dir / "fig2_regime_heatmaps.png", dpi=200)
plt.show()


In [ ]:
# Figure 3: historical stress scenario returns.
if not stress.empty:
    pivot = stress.pivot_table(index="scenario", columns="pair", values="window_return", aggfunc="mean")
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Figure 3. Stress scenario window returns")
    ax.set_ylabel("Window return")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(final_fig_dir / "fig3_stress_window_returns.png", dpi=200)
    plt.show()
else:
    print("Stress table is empty; Figure 3 skipped.")


In [ ]:
# Figure 4: Monte Carlo total-return distributions by pair.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), squeeze=False)
for i, pair in enumerate(["btc_bitx", "qqq_tqqq"]):
    path_file = ROOT / "reports" / "tables" / f"{pair}_mc_path_metrics.csv"
    ax = axes[0, i]

    if not path_file.exists():
        ax.set_title(f"{pair}: no MC file")
        ax.axis("off")
        continue

    df = pd.read_csv(path_file)
    if df.empty:
        ax.set_title(f"{pair}: empty MC file")
        ax.axis("off")
        continue

    ax.hist(df["total_return"], bins=40, alpha=0.75)
    ax.axvline(df["total_return"].quantile(0.05), color="crimson", ls="--", label="5th pct")
    ax.axvline(df["total_return"].quantile(0.50), color="black", ls="--", label="median")
    ax.set_title(f"{pair}: MC total return")
    ax.grid(alpha=0.2)
    ax.legend()

fig.tight_layout()
fig.savefig(final_fig_dir / "fig4_mc_return_distributions.png", dpi=200)
plt.show()


In [ ]:
# Export summary tables that align with the final figure pack.
metrics = pd.read_csv(ROOT / "reports" / "tables" / "baseline_metrics.csv", index_col=0)
stress_table = pd.read_csv(ROOT / "reports" / "tables" / "stress_scenarios.csv")
mc_summary = pd.read_csv(ROOT / "reports" / "tables" / "stress_monte_carlo_summary.csv")

metrics.to_csv(final_tbl_dir / "baseline_metrics.csv")
stress_table.to_csv(final_tbl_dir / "stress_scenarios.csv", index=False)
mc_summary.to_csv(final_tbl_dir / "stress_monte_carlo_summary.csv", index=False)

# Copy per-pair heatmap axis-key tables generated by the regime runner.
for pair in backtests.keys():
    src = ROOT / "reports" / "tables" / f"{pair}_regime_heatmap_key.csv"
    if src.exists():
        pd.read_csv(src).to_csv(final_tbl_dir / src.name, index=False)

print("Saved final tables in:", final_tbl_dir)


In [ ]:
# Quick export sanity checks.
expected_figs = [
    "fig1_equity_curves.png",
    "fig2_regime_heatmaps.png",
    "fig4_mc_return_distributions.png",
]
for fname in expected_figs:
    assert (final_fig_dir / fname).exists(), f"Missing figure: {fname}"

print("Final figure pack sanity checks passed.")


## Suggested Next Steps
- Add benchmark overlays (buy-and-hold spot, static hedge).
- Add cost-sensitivity overlays to show how robust the current edge is to borrow/trading friction.
- Add regime-conditioned Monte Carlo to preserve time-structure effects.
